# Submission v7 — Back to Absolute Features + Surgical Improvements

## What we learned from v1–v6

| Version | LOPO | LB | Key finding |
|---------|------|----|-------------|
| v1 | ~0.35 | **0.354** | Best LB — absolute features + pid_enc + LightGBM |
| v2–v6 | 0.40–0.43 | 0.267–0.308 | z-norm always hurts LB; LOPO stuck regardless |

## v7 strategy

**Keep everything from v1 that worked** (absolute features, pid_enc, LightGBM multi-seed).
**Add only person-invariant features** that don't need normalization:
- `delta` = last-half mean − first-half mean (absolute units, but change is person-invariant)
- `slope` = linear trend across window (same reasoning)
- `t3t1` = last-third mean − first-third mean (finer temporal change)
- HRV features from BPM: SDNN, RMSSD, pNN25, pNN50, LF/HF — ratio/relative by construction

**No z-score normalization at all.**
**Keep pid_enc** — it helps LB even if it hurts LOPO (LOPO ≠ LB in this competition).

Target: LOPO ≥ 0.43 AND LB ≥ 0.354


In [1]:
%pip install lightgbm scikit-learn pandas numpy scipy -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as spstats
from scipy import signal as scsig
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from collections import Counter

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


In [3]:
SENSOR_COLS = ['accel_x','accel_y','accel_z','eda','heart_rate','temperature']
WINDOW_MS  = 180_000
HALF_MS    =  90_000
THIRD_MS   =  60_000

def hrv_features(bpm_series, prefix='hrv'):
    """
    Derive HRV features from instantaneous BPM (person-invariant — ratio/relative features).
    HR updates ~1Hz on Empatica E4 at 32Hz sensor rate. Downsample first.
    """
    f = {}
    keys = ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr','lf_hf','lf_nu','hf_nu']
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in keys: f[f'{prefix}_{k}'] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f[f'{prefix}_sdnn']    = float(np.std(rr))
    f[f'{prefix}_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff)>0 else 0.
    f[f'{prefix}_pnn25']   = float(np.mean(np.abs(rr_diff)>25))*100 if len(rr_diff)>0 else 0.
    f[f'{prefix}_pnn50']   = float(np.mean(np.abs(rr_diff)>50))*100 if len(rr_diff)>0 else 0.
    f[f'{prefix}_mean_rr'] = float(np.mean(rr))
    f[f'{prefix}_cv_rr']   = f[f'{prefix}_sdnn'] / f[f'{prefix}_mean_rr'] if f[f'{prefix}_mean_rr']>1e-6 else 0.
    try:
        freqs, psd = scsig.welch(rr, fs=1.0, nperseg=min(len(rr),64))
        lf = float(np.trapezoid(psd[(freqs>=0.04)&(freqs<0.15)], freqs[(freqs>=0.04)&(freqs<0.15)]))
        hf = float(np.trapezoid(psd[(freqs>=0.15)&(freqs<0.40)], freqs[(freqs>=0.15)&(freqs<0.40)]))
        tot = lf + hf
        f[f'{prefix}_lf_hf'] = lf/hf    if hf>1e-6  else 0.
        f[f'{prefix}_lf_nu'] = lf/tot   if tot>1e-6 else 0.
        f[f'{prefix}_hf_nu'] = hf/tot   if tot>1e-6 else 0.
    except Exception:
        f[f'{prefix}_lf_hf'] = f[f'{prefix}_lf_nu'] = f[f'{prefix}_hf_nu'] = np.nan
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    """
    Extract features using ABSOLUTE (unnormalized) values — same as v1.
    Add delta, slope, t3t1 (person-invariant change features) and HRV features.
    Keep pid_enc (helps LB).
    """
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']; ts = float(lrow['timestamp']); lid = lrow['id']
        feat = {'id': lid}

        if pid not in sensor_by_pid:
            rows.append(feat); continue

        sg = sensor_by_pid[pid]
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<=ts),           SENSOR_COLS]
        wf  = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-HALF_MS),    SENSOR_COLS]
        wl  = sg.loc[(ta>=ts-HALF_MS)  &(ta<=ts),           SENSOR_COLS]
        wt1 = sg.loc[(ta>=ts-WINDOW_MS)&(ta<ts-2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta>=ts-THIRD_MS) &(ta<=ts),           SENSOR_COLS]

        for c in SENSOR_COLS:
            v   = wa[c].dropna().values.astype(float)
            vf  = wf[c].dropna().values.astype(float)
            vl  = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)

            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}'] = np.nan
                continue

            # v1 original features — ABSOLUTE, no normalisation
            feat[f'{c}_mean']   = float(np.mean(v))
            feat[f'{c}_std']    = float(np.std(v))
            feat[f'{c}_min']    = float(np.min(v))
            feat[f'{c}_max']    = float(np.max(v))
            feat[f'{c}_median'] = float(np.median(v))
            feat[f'{c}_skew']   = float(spstats.skew(v))     if len(v)>2 else 0.
            feat[f'{c}_kurt']   = float(spstats.kurtosis(v)) if len(v)>2 else 0.
            feat[f'{c}_range']  = float(np.max(v) - np.min(v))
            feat[f'{c}_q25']    = float(np.percentile(v, 25))
            feat[f'{c}_q75']    = float(np.percentile(v, 75))
            feat[f'{c}_iqr']    = float(np.percentile(v,75) - np.percentile(v,25))

            # Change features — person-invariant (baseline cancels in subtraction)
            feat[f'{c}_delta'] = float(np.mean(vl) - np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.
            feat[f'{c}_slope'] = float(np.polyfit(np.linspace(0,1,len(v)), v, 1)[0]) if len(v)>2 else 0.
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1)>0 else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3)>0 else float(np.mean(v))
            feat[f'{c}_t3t1']    = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']

        # Accel magnitude (v1 feature)
        ax=wa['accel_x'].values; ay=wa['accel_y'].values; az=wa['accel_z'].values
        if len(ax)>0:
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan

        # HRV features — ratio-based, person-invariant (no normalization needed)
        feat.update(hrv_features(wa['heart_rate']))

        # pid_enc — keeps v1's person identity signal that helped LB
        feat['pid_enc'] = pid_enc_map.get(pid, -1)

        rows.append(feat)
    return pd.DataFrame(rows).set_index('id')

# Build pid encoder from TRAIN only (test PIDs map to -1 = unknown person)
train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
print(f'  shape: {train_features.shape}')

print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, train_pid_map)
print(f'  shape: {test_features.shape}')
print()
print('Feature groups:')
print(f'  v1 stats (11 per sensor): {11*6} = 66')
print(f'  change features (5 per sensor): {5*6} = 30')
print(f'  accel mag: 3')
print(f'  HRV: 9')
print(f'  pid_enc: 1')
print(f'  Total: {66+30+3+9+1}')


Extracting train features...
  shape: (815, 109)
Extracting test features...
  shape: (1028, 109)

Feature groups:
  v1 stats (11 per sensor): 66 = 66
  change features (5 per sensor): 30 = 30
  accel mag: 3
  HRV: 9
  pid_enc: 1
  Total: 109


In [4]:
tli    = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {c: total/(n_cls*cnt) for c,cnt in counts.items()}
sample_weights = np.array([class_weights[yi] for yi in y])

print('X_imp shape     :', X_imp.shape)
print('X_test_imp shape:', X_test_imp.shape)
print('Class distribution:', dict(counts))
print('Class weights:', {k: round(v,3) for k,v in class_weights.items()})


X_imp shape     : (815, 109)
X_test_imp shape: (1028, 109)
Class distribution: {1: 66, 0: 162, 2: 587}
Class weights: {1: 4.116, 0: 1.677, 2: 0.463}


In [5]:
# v1 LightGBM hyperparameters — kept exactly as original best
LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.02,
    num_leaves        = 127,
    max_depth         = -1,
    min_child_samples = 5,
    subsample         = 0.6,
    colsample_bytree  = 0.6,
    reg_alpha         = 0.3,
    reg_lambda        = 0.3,
    class_weight      = 'balanced',
    objective         = 'multiclass',
    num_class         = 3,
    n_jobs            = -1,
    verbose           = -1,
)

print('=== LOPO CV — v7 (absolute + change + HRV + pid_enc) ===')
logo = LeaveOneGroupOut()
lopo_scores = []

for tr_idx, val_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[val_idx[0]]
    y_val   = y.iloc[val_idx]
    if len(y_val.unique()) < 2:
        print(f'  Skip {pid_val}: only 1 class'); continue

    m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': 42})
    m.fit(X_imp.iloc[tr_idx], y.iloc[tr_idx],
          sample_weight = sample_weights[tr_idx],
          eval_set      = [(X_imp.iloc[val_idx], y_val)],
          callbacks     = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])

    sc = balanced_accuracy_score(y_val, m.predict(X_imp.iloc[val_idx]))
    print(f'  Leave out {pid_val}: {sc:.4f}  (n={len(val_idx)}, classes={sorted(y_val.unique())})')
    lopo_scores.append(sc)

print(f'\nv7 LOPO = {np.mean(lopo_scores):.4f} +/- {np.std(lopo_scores):.4f}')
print('v1 ref   = ~0.35 est  |  v4 ref (best LOPO) = 0.4288')
print()
print('Decision rule:')
print('  If LOPO > 0.43 → this is a real improvement, submit')
print('  If LOPO ≈ 0.40-0.43 → similar to v4, worth submitting (LB likely ≥ v1 since no z-norm)')
print('  If LOPO < 0.40 → revert to v1 and tune hyperparams only')


=== LOPO CV — v7 (absolute + change + HRV + pid_enc) ===
  Leave out 43JW: 0.5000  (n=93, classes=[np.int64(0), np.int64(2)])
  Leave out C8Q6: 0.4894  (n=152, classes=[np.int64(0), np.int64(2)])
  Leave out DT5C: 0.5670  (n=90, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out F1ZM: 0.4888  (n=137, classes=[np.int64(1), np.int64(2)])
  Leave out HDS9: 0.6303  (n=135, classes=[np.int64(0), np.int64(2)])
  Leave out P4DZ: 0.3197  (n=144, classes=[np.int64(0), np.int64(1), np.int64(2)])
  Leave out TPQI: 0.5908  (n=64, classes=[np.int64(0), np.int64(2)])

v7 LOPO = 0.5123 +/- 0.0937
v1 ref   = ~0.35 est  |  v4 ref (best LOPO) = 0.4288

Decision rule:
  If LOPO > 0.43 → this is a real improvement, submit
  If LOPO ≈ 0.40-0.43 → similar to v4, worth submitting (LB likely ≥ v1 since no z-norm)
  If LOPO < 0.40 → revert to v1 and tune hyperparams only


In [6]:
print('=== Final ensemble: 3 seeds x 5 folds ===')
SEEDS = [42, 7, 123]
all_test_proba = []
all_cv_scores  = []

for seed in SEEDS:
    skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
        X_tr = X_imp.iloc[tr_idx]; y_tr = y.iloc[tr_idx]
        X_va = X_imp.iloc[val_idx]; y_va = y.iloc[val_idx]
        sw_tr = sample_weights[tr_idx]

        m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m.fit(X_tr, y_tr,
              sample_weight = sw_tr,
              eval_set      = [(X_va, y_va)],
              callbacks     = [lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])

        sc = balanced_accuracy_score(y_va, m.predict(X_va))
        fold_scores.append(sc)
        seed_proba += m.predict_proba(X_test_imp)
        print(f'  Seed {seed} Fold {fold+1}: val BA = {sc:.4f}')

    seed_proba /= 5
    all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed} mean CV = {np.mean(fold_scores):.4f}')

print(f'\nEnsemble CV = {np.mean(all_cv_scores):.4f}')
print('v1 reference = 0.8234')

final_proba = np.mean(all_test_proba, axis=0)
final_preds = np.argmax(final_proba, axis=1).astype(int)

print('\nPrediction distribution:')
for u, cnt in zip(*np.unique(final_preds, return_counts=True)):
    print(f'  class {u}: {cnt}  ({cnt/len(final_preds)*100:.1f}%)')

print('\nTrain distribution for reference:')
for cls, cnt in Counter(y).items():
    print(f'  class {cls}: {cnt}  ({cnt/len(y)*100:.1f}%)')


=== Final ensemble: 3 seeds x 5 folds ===
  Seed 42 Fold 1: val BA = 0.8414
  Seed 42 Fold 2: val BA = 0.7708
  Seed 42 Fold 3: val BA = 0.8469
  Seed 42 Fold 4: val BA = 0.7622
  Seed 42 Fold 5: val BA = 0.8524
  Seed 42 mean CV = 0.8147
  Seed 7 Fold 1: val BA = 0.8415
  Seed 7 Fold 2: val BA = 0.8350
  Seed 7 Fold 3: val BA = 0.8169
  Seed 7 Fold 4: val BA = 0.8569
  Seed 7 Fold 5: val BA = 0.7821
  Seed 7 mean CV = 0.8265
  Seed 123 Fold 1: val BA = 0.8642
  Seed 123 Fold 2: val BA = 0.6812
  Seed 123 Fold 3: val BA = 0.8809
  Seed 123 Fold 4: val BA = 0.8191
  Seed 123 Fold 5: val BA = 0.7952
  Seed 123 mean CV = 0.8081

Ensemble CV = 0.8164
v1 reference = 0.8234

Prediction distribution:
  class 0: 331  (32.2%)
  class 1: 581  (56.5%)
  class 2: 116  (11.3%)

Train distribution for reference:
  class 1: 66  (8.1%)
  class 0: 162  (19.9%)
  class 2: 587  (72.0%)


In [7]:
submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': final_preds})
submission.to_csv('submission_v7.csv', index=False)
print('submission_v7.csv saved!')
print(submission.head(10))


submission_v7.csv saved!
     id  stress
0  1227       2
1  1228       0
2  1229       2
3  1230       2
4  1231       1
5  1232       0
6  1233       0
7  1234       2
8  1235       0
9  1236       1
